# Lecture 4 — Modules, Recursion, and a Simple Checkers AI

## Why Python? (An Aside)

Before diving in, it's worth pausing on a question that sits underneath this whole course: why is Python *the* language of data science?

**Interpreted vs. compiled.** With a compiled language, you write code into files, a compiler turns it into machine code, and you run the machine code. That's fast, and many errors (syntax mistakes, certain logic errors) get caught at compile time, before the program ever runs. With an interpreted language like Python, a running program ingests your code and does what it says — which means an error sitting in a line of code isn't discovered until that line actually executes.

**Why interpreted wins for data work.** When you're playing with data, you want to try something, see the result, adjust, and try again — *without losing your context*. In a compiled workflow, every experiment means recompiling and re-running from scratch. In a notebook, you can have done a hundred steps and then iterate on step 101 five or six times until it looks right. A practical tip that follows: when you need to do something complicated and don't know how, open a scratch cell, experiment until the piece works, and only then drop it into your real code.

**Why slowness doesn't matter: Python is a glue language.** Practically every serious software package, whatever language it's written in, exposes a Python API. So Python is the *glue*: you set up the problem and steer, while the heavy computation happens inside compiled libraries. If you take the dot product of two million-element vectors with a Python loop, it takes forever; hand the same vectors to `numpy` and the computation runs in compiled code — and the deep-learning packages push it further down, onto accelerators via the manufacturer's own libraries. Say *what* to compute in Python, and let numpy and the hardware vendors figure out how to do it fast.

**Other reasons Python fits:**

- **Easy, powerful data structures.** Lists and especially dictionaries let you assemble complicated data on the fly, without designing custom data structures first.
- **Memory management.** Python does reference counting and garbage collection: when nothing refers to a piece of memory anymore, Python frees it. You (mostly) never think about it.
- **Dynamic typing.** The same line `x * y` works whether `x` and `y` are numbers, numpy arrays, or TensorFlow tensors — the code does different things at runtime depending on what the data is. That's powerful (you can swap backends without rewriting code) but it's a trade-off: a type mistake may silently compute something you didn't intend instead of raising an error. You wouldn't want to write an operating system this way; for data science the flexibility usually wins. It does mean you need to keep a little Python interpreter running in your head, so when a number looks wrong you can reason about what your code actually did.
- **All the paradigms.** Python supports structured programming (what we're doing now), functional programming (coming soon), and object-oriented programming (later in the course).

You will also run into **R** in other courses. R is a statistics package with a language on top of it, and it's very good at what it does — but there's little it does that you can't also do in Python, which is part of why Python keeps growing.

Finally, a skill to cultivate: **reading code**. If variables are named well, Python reads almost like English — and as a data scientist you'll be confronted with other people's code all the time. Reading it is like reading books: it's also how you learn to write.

## Python Libraries and Modules (Review)

One of the great things about **Python** is its ecosystem of libraries. Many are already installed (the *standard library*), and many more are available on the internet.

Two good starting points:

- [The Python Standard Library reference](https://docs.Python.org/3/library/index.html): packages that ship with Python.
- [PyPI](https://pypi.org): third‑party packages you can install with `pip install ...`.

### Python's `import`

Jupyter notebooks execute code when you run a cell. All variables and function definitions live in the Python *interpreter session* behind the notebook.

In a real software project, code is usually organized into **files** (modules). For example, we can put the checkers code from Lecture 3 into a file called `checkers.py`, and then import it here.

**Setup note:** This notebook assumes `checkers.py` is in the **same folder** as this notebook (or otherwise on your Python path). If you see `ModuleNotFoundError: No module named 'checkers'`, copy/download `checkers.py` into this directory and rerun the import cell below.


### Why modules?

Here's how modules fit the data-science workflow. You start a notebook, open up some data, and begin writing functions — to load it, clean it, massage it into shape. Then you start the *next* notebook, which continues from those steps. You do **not** want to copy and paste the functions over.

A very common failure mode for students (and working data scientists): the notebook grows into a novel — hundreds of cells — and to "run" it you execute this cell, then that one, out of order. That is not reproducible: next time through you do it slightly differently and get a different answer, and you won't know why.

The better practice: once you've solved a problem — say, loading and cleaning your data — move those functions into a **module** (a `.py` file) and `import` it. The solved problem becomes an abstraction; the next notebook starts clean with one import line. That's exactly what we've done with last lecture's checkers code: all the functions we wrote in cells are now copied into a file `checkers.py` that we can import.

In [1]:
import checkers

We now have a "checkers" object in our current context:

In [2]:
print(checkers)
print(type(checkers))

<module 'checkers' from '/Users/dr.farbin/Library/CloudStorage/Dropbox/Teaching/Spring-26-3402/DATA3402.Spring.2026/Lectures/Lecture.4/checkers.py'>
<class 'module'>


We can see the contents of the module using the `dir` built-in:

In [3]:
dir(checkers)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'checkers_game',
 'column_map',
 'column_names',
 'count_pieces',
 'draw_board',
 'empty',
 'empty_space',
 'game_won',
 'get_size',
 'left_move',
 'make_game_board',
 'move_piece',
 'moves',
 'nice_move_piece',
 'parse_location',
 'parse_move',
 'player_1',
 'player_1_left_move',
 'player_1_piece',
 'player_1_right_move',
 'player_2',
 'player_2_left_move',
 'player_2_piece',
 'player_2_right_move',
 'player_moves',
 'print_message',
 'right_move',
 'row_map',
 'row_names',
 'size',
 'space_character',
 'switch_player',
 'take_move']

Note that calling `dir()` without an argument will show you everything in your current context:

In [4]:
x=1
dir()

['In',
 'Out',
 '_',
 '_3',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__session__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'checkers',
 'exit',
 'get_ipython',
 'open',
 'quit',
 'x']

To call something from the checkers module, we simply do, for example:

In [5]:
checkers.make_game_board()

[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 1, 0, 1, 0, 1, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 2, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

This may be cumbersome, so we can import a specific part of the checkers into our current context:

In [6]:
from checkers import make_game_board
print(make_game_board())
dir()

[[0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [2, 0, 2, 0, 2, 0, 2, 0], [0, 2, 0, 2, 0, 2, 0, 2], [2, 0, 2, 0, 2, 0, 2, 0]]


['In',
 'Out',
 '_',
 '_3',
 '_4',
 '_5',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__session__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'checkers',
 'exit',
 'get_ipython',
 'make_game_board',
 'open',
 'quit',
 'x']

We can rename what we import in our current context:

One place renaming really pays off is swapping *backends*. All of the major tensor libraries (numpy, TensorFlow, PyTorch, ...) provide nearly the same interface to arrays, even though the implementations underneath are completely different. So code developed against numpy can often be switched over to TensorFlow just by changing *which* package gets imported under the common name — the rest of the code never has to change.

In [7]:
from checkers import moves as checkers_moves
print(checkers_moves)

{1: {0: (1, 1), 1: (1, -1)}, 2: {0: (-1, -1), 1: (-1, 1)}}


Or we can import everything into our current context using `*`:

In [8]:
from checkers import *
dir()

['In',
 'Out',
 '_',
 '_3',
 '_4',
 '_5',
 '_6',
 '__',
 '___',
 '__builtin__',
 '__builtins__',
 '__doc__',
 '__loader__',
 '__name__',
 '__package__',
 '__session__',
 '__spec__',
 '_dh',
 '_i',
 '_i1',
 '_i2',
 '_i3',
 '_i4',
 '_i5',
 '_i6',
 '_i7',
 '_i8',
 '_ih',
 '_ii',
 '_iii',
 '_oh',
 'checkers',
 'checkers_game',
 'checkers_moves',
 'column_map',
 'column_names',
 'count_pieces',
 'draw_board',
 'empty',
 'empty_space',
 'exit',
 'game_won',
 'get_ipython',
 'get_size',
 'left_move',
 'make_game_board',
 'move_piece',
 'moves',
 'nice_move_piece',
 'open',
 'parse_location',
 'parse_move',
 'player_1',
 'player_1_left_move',
 'player_1_piece',
 'player_1_right_move',
 'player_2',
 'player_2_left_move',
 'player_2_piece',
 'player_2_right_move',
 'player_moves',
 'print_message',
 'quit',
 'right_move',
 'row_map',
 'row_names',
 'size',
 'space_character',
 'switch_player',
 'take_move',
 'x']

## Reloading Modules

Note that you can change contents of the libraries that you load:

In [9]:
checkers.size

8

In [10]:
checkers.size=10

In [11]:
print(checkers.size)

10


In [12]:
checkers.get_size()

10

In general, changing values inside an imported module like this isn’t good practice.

But it illustrates an important behavior: **`import` does not re-run a module’s code if it has already been imported** in the current Python session.


In [13]:
import checkers

In [14]:
print(checkers.size)

10


In [15]:
checkers.get_size()

10

However, you can explicitly reload a module if you need to pick up changes on disk:

In [16]:
import importlib
importlib.reload(checkers)
checkers.size

8

In [17]:
checkers.size

8

In [18]:
checkers.get_size()

8

### A note on editing modules and reproducibility in notebooks

It’s important to remember that when you are working interactively in a notebook (or any interactive Python session) and you **edit a module file**, Python does *not* automatically pick up those changes. To see your updates, you must explicitly reload the module and rerun any code that depends on it.

More generally, **before showing results or drawing conclusions, you should re-run your entire notebook from top to bottom**. This is true even if you haven’t edited any module files.

As you work through a data science problem, you will naturally:
- edit cells,
- rerun some cells multiple times,
- skip others,
- and change earlier code after later cells have already executed.

It is very common for the *current state* of a notebook to reflect code that was executed **out of order**, or with definitions that have since changed. When this happens, the results you are looking at may no longer correspond to the code you think produced them.

For that reason, re-running the notebook from a clean state is essential to ensure that:
- your results are reproducible,
- your conclusions are based on the code as it currently exists,
- and you are not accidentally relying on stale or inconsistent state.

This habit is a key part of responsible and reliable data science practice.

## Recursion

At the end of today’s lecture we’ll use a **tree search** algorithm to implement a computer checkers player. We’ll use recursion to tackle that problem, so let’s take a bit of time to review recursion first.

A **recursive function** is one that calls itself. Some problems are simpler to express recursively because the solution naturally breaks into “a smaller version of the same problem”.

A classic example is factorial:

$$
n! = n \cdot (n-1)\cdot(n-2)\cdot \dots \cdot 2 \cdot 1
$$

We can write this recursively by noticing:

- $n! = n \cdot (n-1)!$
- Base case: $1! = 1$

Let’s code it up:

Why introduce recursion *now*? Because of where we're headed. To pick a move, our computer player will consider every possible move, then every possible response, then every move after that — a **tree of possibilities** that grows exponentially with depth. Trees are exactly the kind of structure that recursion handles gracefully: to process a tree, you do the same thing to each branch that you did to the whole. Traversing a tree any other way is painful, as we'll see at the end of this section.

In [19]:
def factorial(n):
    if n == 1:
        return 1
    else:
        return n * factorial(n-1)

In [20]:
factorial(50)

30414093201713378043612608166064768844377641568960512000000000000

We can see how Python is performing the computation by *not* multiplying, and instead returning the intermediate values in a tuple:

Watch the shape of what happens below: the calls build a nested structure *down* until they hit the termination (`factorial(1)` returning `1`), and only then do the actual multiplications happen on the way back *up*, as each call returns to the one that made it. That is the typical anatomy of a recursive computation: build the structure, reach the base case, then combine the results as the calls unwind. The tuple trick isn't a real computation — it's just a way to *see* the order in which everything happens.

In [21]:
def factorial_0(n):
    if n == 1:
        return 1
    else:
        return (n , factorial_0(n-1))

In [22]:
factorial_0(5)

(5, (4, (3, (2, 1))))

Note that you don't need to use recursion to compute factorial:

In [23]:
def factorial_1(n):
    prod = 1
    for e in range(1,n+1):
        prod*=e  
    return prod

factorial_1(50)

30414093201713378043612608166064768844377641568960512000000000000

But sometimes it's much simpler to think through the implementation recursively.

### Loops in recursion
There are various ways to implement loops recursively, for example:

In [24]:
def countdown(n):
     print(n)
     if n == 0:
         return             # Terminate recursion
     else:
         countdown(n - 1)   # Recursive call
countdown(10)

10
9
8
7
6
5
4
3
2
1
0


Here is an implementation of Python's `range` function using recursion:

In [25]:
def rec_range(start,stop=None,step=1):
    # Properly handle default arguments
    if stop:
       pass
    else:
        stop=start
        start=0
        
    if start < stop:
        return [start] + rec_range(start+step,stop,step)
    else:
        return []

In [26]:
rec_range(2,20,2)

[2, 4, 6, 8, 10, 12, 14, 16, 18]

A recursive function must include a conditional that separates two cases:

- a case where the function **calls itself** (with slightly changed arguments), and
- a case where the function **stops** and returns a result.

This stopping case is called the **base case**. Without a base case, the function would keep calling itself forever.

When you are designing a recursive algorithm, make sure you can clearly answer the following questions:
- What is the **base case** that ends the recursion?
- What value should be **returned** in that base case?
- How do the function arguments change in each recursive call?
- How are the results **combined or accumulated** as the recursion returns back to the original function call?

If you can answer these questions, your recursive solution will usually be both correct and easier to reason about.

### More recursive examples
Another algorithm that is often implemented recursively is the Fibonacci sequence $F_0=0$, $F_1=1$, $F_n = F_{n-1} + F_{n-2}$. Here an implementation:

In [27]:
def recur_fibo(n):
   if n <= 1:
       return n
   else:
       return(recur_fibo(n-1) + recur_fibo(n-2))

Checking for a palindrome:

In [28]:
def is_palindrome(word):
     if len(word) <= 1:
         return True
     else:
         return word[0] == word[-1] and is_palindrome(word[1:-1])

Of course, there are other ways to do the same thing:

In [29]:
def is_palindrome_0(word):
    return word == word[::-1]

### Recursion Limit
Be aware that recursion isn't free: every nested call takes memory, and *runaway* recursion — a missing or wrong base case — will consume it until things crash. To protect you from that, Python limits how many times a function can call itself. The default limit is 1000, which is actually not very much:

In [30]:
from sys import getrecursionlimit
getrecursionlimit()
1000

1000

Which you can increase if you need:

In [31]:
from sys import setrecursionlimit
setrecursionlimit(2000)
getrecursionlimit()
2000

2000

### Traversing a Tree
The reason we are looking at recursion now, is that is the easiest way to traverse tree-like data structures. Consider the following data structure:

In [32]:
data = [1 ,[21,22], [23,[31,32]], [[33,34],[35,36]] ]
data

[1, [21, 22], [23, [31, 32]], [[33, 34], [35, 36]]]

Imagine you needed to count how many numbers are stored in the structure (aka count the leaves in the tree). How would you do it?

Here's an approach:
* Create a `leaf_count` function that takes a list as argument.
* In the function, keep a counter, initialized to 0.
* Iterate over the list. 
    * If you see a number add one to counter.
    * If you see a list, call `leaf_count` with that list, add results to counter.
* Return the counter value.


In [33]:
def leaf_count(ll):
    count = 0
    for item in ll:
        if isinstance(item, list):
            count += leaf_count(item)
        else:
            count += 1

    return count

In [34]:
leaf_count(data)

10

Performing the same operation non-recursively is much more complicated:

Try to read the version below and convince yourself it's correct. It's hard — and that's exactly the point. You *can* always avoid recursion, but for tree-shaped problems the non-recursive version is dramatically harder to write, read, and get right.

In [35]:
def leaf_count_non_recursive(ll):
    count = 0
    stack = []
    current_ll = ll
    i = 0

    while True:
        if i == len(current_ll):
            if current_ll == ll:
                return count
            else:
                current_ll, i = stack.pop()
                i += 1
                continue

        if isinstance(current_ll[i], list):
            stack.append([current_ll, i])
            current_ll = current_ll[i]
            i = 0
        else:
            count += 1
            i += 1

In [36]:
leaf_count_non_recursive(data)

10

## Checkers "AI" Player

Let's now attempt to build an **AI checkers player** to play against a human.

First, it's worth asking: how do *humans* do it? When you play a game like chess or checkers, you think a few moves ahead — but a strong player doesn't grind through every possibility. They recognize **patterns**: "this position is bad," "that piece is exposed." Experience gives them shortcuts, so they only think deeply about the lines that matter.

Computers historically took the opposite route: **brute force**. A computer is fast, so let it consider *all* possible moves, all possible responses, and so on — building a tree of possibilities — and then pick the branch with the best outcomes. A famous milestone: in the 1990s, IBM's Deep Blue supercomputer beat the world's best chess player, fundamentally by searching the game tree deeply and cleverly enough to out-calculate a human. The game of Go resisted this approach for decades — its tree of possibilities grows so much faster than chess that no amount of hardware could brute-force it, and many people believed a computer would never beat the best human players. That belief fell when DeepMind's AlphaGo beat the world's best Go player, using *machine learning* rather than exhaustive search.

The same trade-off shows up outside of games. (An aside from research: in experimental physics we have classical algorithms for understanding detector data whose cost blows up as the accelerator's intensity is turned up — one reason we turn to machine learning is that the learned version of the algorithm *scales* far better than the exhaustive one.)

So there are two broad paths to a game-playing AI:

1. **Game-tree search (what we will do here):**
   From a given board, generate possible moves, then possible responses, then moves after that, and so on. These possibilities form a **tree**. The number of possibilities grows roughly exponentially as you look further ahead, so depth matters.

2. **Machine learning (not our focus today):**
   For example, reinforcement learning can produce strong game-playing agents by learning from experience.

For our example, we'll use a basic game-tree search.

First, we need a way to **score** a board position: after some sequence of moves, did I end up in a better place or a worse one? A simple (not necessarily optimal) scoring function is to count my pieces and subtract off my opponent's:

In [37]:
# A scoring function
def score_board(board,player):
    return count_pieces(board,player)-count_pieces(board,switch_player(player))

Next, let’s sketch the algorithm.

We want a function that, starting from the current board, **generates and scores** possible move sequences up to some maximum depth. This is naturally recursive.

**Inputs to the function:**
- the current `board`
- the `player_to_win` (who we are trying to help)
- the `current_player` (whose turn it is at this node of the tree)
- a remaining `depth`

**Outputs:**
- a list of move sequences (or move chains)
- a matching list of scores

High-level logic:

- If `depth == 0`, stop expanding the tree and **score the board**.
- Otherwise:
  - Find all pieces for `current_player`
  - For each piece, consider each possible direction (left/right)
    - If the move is legal, apply it (on a copy of the board)
    - Recursively score the resulting position with `depth - 1`

Note the payoff of a design decision from last lecture: `move_piece` already contains *all* of the rules of the game, returns `True`/`False` for whether a move is legal, and can be told to keep quiet with `verbose=False`. So the AI can "think about" a move simply by trying it on a copy of the board — we reuse the function completely unchanged.

One more detail in the implementation below: the `keep_all_moves` option. To actually *choose* a move, we only need each possible **first** move and the scores at the ends of the chains — once the opponent responds, we'll re-run the search from the new position anyway. So by default we don't store every intermediate move; it's a memory-and-time trade-off. We'll turn it on below so we can inspect what the search is doing.

In [38]:
import copy

def generate_moves(board,player,current_player,depth, keep_all_moves=False):
    if depth==0:
        return list(),score_board(board,player)

    moves=list()
    scores=list()
    
    # Look through the board for pieces belonging to a specific player
    for i in range(size):
        for j in range(size):
            if board[i][j]==current_player:
                
                # For each piece, consider moving left or right
                for move in [left_move,right_move]:
                    
                    # Create a new copy of the board
                    new_board=copy.deepcopy(board)
                    
                    # Check if its a valid move
                    if move_piece(new_board,current_player,(i,j),move,verbose=False):
                        # If so, generate subsequent moves
                        next_moves,score=generate_moves(new_board,player,
                                                        switch_player(current_player),
                                                        depth-1,
                                                        keep_all_moves=keep_all_moves)
                        
                        # Keep track of this move
                        this_move=[(current_player,(i,j),move)]
                        if keep_all_moves:
                            # For our purposes, we usually only need to keep the top move!
                            this_move.extend(next_moves)
                        moves.append(this_move)
                        scores.append(score)

    return moves,scores
            


### Quick side on use of `deepcopy`

Why do we need a copy at all? Because `move_piece` *modifies the board we hand it* — lists are passed by reference. That was exactly what we wanted when playing the game, but the AI only wants to **consider** a move, not commit to it. If we passed in the real board, every hypothetical move would destroy the actual game state. So we hand `move_piece` a copy and let it scribble on that.

And a plain `copy.copy` isn't enough: it copies only one layer — the outer list — while the row lists inside are still shared references. `copy.deepcopy` recursively duplicates every nested list, which is what we need for a list-of-lists. Consider the following examples.

In [39]:
# Assignment doesn't work
x = [[1,2],[2,3]]
y = x
y[0][1]=-1
x

[[1, -1], [2, 3]]

In [40]:
# Copy works, but ...
x = [1,2,3]
y = copy.copy(x)
y[0]=-1
x

[1, 2, 3]

In [41]:
# Copy doesn't work for nested lists
x = [[1,2],[2,3]]
y = copy.copy(x)
y[0][1]=-1
x

[[1, -1], [2, 3]]

In [42]:
# Why use deep copy?
x = [[1,2],[2,3]]
y = copy.deepcopy(x)
y[0][1]=-1
x

[[1, 2], [2, 3]]

### Test `generate_moves`

Let's see what generate moves does. Let's make a board:

In [43]:
from checkers import *
board=make_game_board()
draw_board(board)

  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C   X   X   X   X 
D                 
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   


Now create all possible first moves for player 1, scoring to depth 1. We'll keep all subsequent moves so we can inspect them.

In [44]:
possible_moves,scores=generate_moves(board,player_1,player_1,1,keep_all_moves=True)

Here are the possible moves:

In [45]:
possible_moves

[[(1, (2, 1), 0)],
 [(1, (2, 1), 1)],
 [(1, (2, 3), 0)],
 [(1, (2, 3), 1)],
 [(1, (2, 5), 0)],
 [(1, (2, 5), 1)],
 [(1, (2, 7), 1)]]

We see seven possible first moves. Let's increase depth to see what happens:

In [46]:
possible_moves,scores=generate_moves(board,player_1,player_1,2,keep_all_moves=True)

In [47]:
possible_moves

[[(1, (2, 1), 0),
  [(2, (5, 0), 1)],
  [(2, (5, 2), 0)],
  [(2, (5, 2), 1)],
  [(2, (5, 4), 0)],
  [(2, (5, 4), 1)],
  [(2, (5, 6), 0)],
  [(2, (5, 6), 1)]],
 [(1, (2, 1), 1),
  [(2, (5, 0), 1)],
  [(2, (5, 2), 0)],
  [(2, (5, 2), 1)],
  [(2, (5, 4), 0)],
  [(2, (5, 4), 1)],
  [(2, (5, 6), 0)],
  [(2, (5, 6), 1)]],
 [(1, (2, 3), 0),
  [(2, (5, 0), 1)],
  [(2, (5, 2), 0)],
  [(2, (5, 2), 1)],
  [(2, (5, 4), 0)],
  [(2, (5, 4), 1)],
  [(2, (5, 6), 0)],
  [(2, (5, 6), 1)]],
 [(1, (2, 3), 1),
  [(2, (5, 0), 1)],
  [(2, (5, 2), 0)],
  [(2, (5, 2), 1)],
  [(2, (5, 4), 0)],
  [(2, (5, 4), 1)],
  [(2, (5, 6), 0)],
  [(2, (5, 6), 1)]],
 [(1, (2, 5), 0),
  [(2, (5, 0), 1)],
  [(2, (5, 2), 0)],
  [(2, (5, 2), 1)],
  [(2, (5, 4), 0)],
  [(2, (5, 4), 1)],
  [(2, (5, 6), 0)],
  [(2, (5, 6), 1)]],
 [(1, (2, 5), 1),
  [(2, (5, 0), 1)],
  [(2, (5, 2), 0)],
  [(2, (5, 2), 1)],
  [(2, (5, 4), 0)],
  [(2, (5, 4), 1)],
  [(2, (5, 6), 0)],
  [(2, (5, 6), 1)]],
 [(1, (2, 7), 1),
  [(2, (5, 0), 1)],
  [(2, (

In [48]:
len(possible_moves)

7

We see that for each of the seven first moves, there are seven responses by the other player.

Now let's look at the associated scores (remember depth is 2):

In [49]:
scores

[[0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0]]

For each of the seven first moves and seven responses, no pieces are taken, so score is 0 in all scenarios.

Let's increase depth to 3 and see what happens:

In [50]:
possible_moves,scores=generate_moves(board,player_1,player_1,3,keep_all_moves=True)
scores

[[[0, 0, 0, 0, 0, 0, 0, 1],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 1],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 1],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0, 0, 1],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 1, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 1],
  [0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0]],
 [[0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0,

Now we see that there are several first moves that could result in a positive score, if the other player makes a bad move.

Let's look at depth 5:

(Notice that the depth-5 call takes visibly longer to run. The tree grows exponentially with depth — this is the fundamental constraint on brute-force search. We cannot ask the computer to look 100 moves ahead; it would simply never finish.)

In [51]:
possible_moves,scores=generate_moves(board,player_1,player_1,5)
scores

[[[[[0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0]],
   [[0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 0]],
   [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [-1, -1, 0, -1, -1, -1, -1],
    [0, 0, 0, 0, 0, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0]],
   [[0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]],
   [[0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 1],
    [0, 0, 0, 0, 0

### Selecting a move

We now need an algorithm that looks at all these scenarios and selects one.

The number of possibilities is large. Some move chains are inconsequential (score 0), while others are great (> 0) or horrible (< 0). Some outcomes happen sooner than others, and it can make sense to weigh earlier outcomes more heavily.

Here is one simple proposal:

- At the end of the move chain, consider both the **best** and **worst** outcomes.
- Otherwise, combine scores over all possibilities, weighting by depth.

Here is a recursive implementation of that idea:

Why weight by depth? A capture available in two moves is worth more than the same capture five moves out — the further away an outcome is, the more it depends on the opponent cooperating, so we discount it.

Note also that *summing* over outcomes is itself a strategy choice. Instead of adding everything up, we could propagate only the **best**-case outcome down each branch, or only the **worst**-case (what should I expect if my opponent plays perfectly?). These give genuinely different players, and none of them is obviously "right" — this is a place where you can experiment. (Doing the search in this repeated best/worst fashion is essentially the *minimax* idea mentioned in the summary.)

In [52]:
def tree_search_0(t, depth=1):
    """Example weighting function: average outcomes, discounted by depth."""
    if isinstance(t[0], list):
        return sum([tree_search_0(item, depth + 1) / depth for item in t])
    else:
        return sum(t)

def tree_search(t, depth=1):
    """Heuristic: combine best and worst outcomes at the leaves, discounted by depth."""
    if isinstance(t[0], list):
        return sum([tree_search(item, depth + 1) / depth for item in t])
    else:
        return max(t) + min(t)


Quick test:

In [53]:
list(map(tree_search,scores))

[27.0,
 28.833333333333332,
 26.333333333333332,
 26.333333333333332,
 24.5,
 27.666666666666664,
 23.666666666666664]

Finally a function to put it all together and select a move:

In [58]:
def pick_move(board,player,depth=5,func=max):
    moves,scores=generate_moves(board,player,player,depth)
    result=list(map(tree_search,scores))
    move_index=result.index(func(result))
    return moves[move_index][0]

Let's test:

In [55]:
pick_move(board,player_1)

(1, (2, 1), 1)

Finally put it all together:

In [59]:
def checkers_game_AI(depth=5):
    
    print ("Welcome to Checkers.")
    print ("--------------------")

    # Make a game board
    board_0=make_game_board()
    
    # Start with player 1
    player=player_1
    
    this_game_won=False
    while not this_game_won:
        # Draw the board
        draw_board(board_0)
        
        # Make a move
        if player==player_1:
            print("Player",player,"move:")
            take_move(board_0,player)
        else:
            the_move=pick_move(board_0,player_2,depth=depth)
            #print(the_move)
            move_piece(board_0,*the_move)
            
        # Check if the game has been won
        this_game_won=game_won(board_0)

       # Switch players
        player=switch_player(player)          

    print("Player 1 Pieces:", count_pieces(board_0,player_1))
    print("Player 2 Pieces:", count_pieces(board_0,player_2))
    
    if this_game_won:
        print("Winner is player:",this_game_won)

In [ ]:
# OPTIONAL (interactive): play against the AI
RUN_AI_GAME = True  # set to True to enable

if RUN_AI_GAME:
    checkers_game_AI()


Welcome to Checkers.
--------------------
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C   X   X   X   X 
D                 
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   
Player 1 move:


Input location: C2
Input move (L/R): L


Moved.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C       X   X   X 
D     X           
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   
Moved.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C       X   X   X 
D     X           
E               O 
F O   O   O       
G   O   O   O   O 
H O   O   O   O   
Player 1 move:


Input location: D3
Input move (L/R): L


Moved.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C       X   X   X 
D                 
E       X       O 
F O   O   O       
G   O   O   O   O 
H O   O   O   O   
Took opponent's piece.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C       X   X   X 
D     O           
E               O 
F O   O           
G   O   O   O   O 
H O   O   O   O   
Player 1 move:


Input location: C4
Input move (L/R): R


Took opponent's piece.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C           X   X 
D                 
E   X           O 
F O   O           
G   O   O   O   O 
H O   O   O   O   
Took opponent's piece.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C           X   X 
D     O           
E               O 
F     O           
G   O   O   O   O 
H O   O   O   O   
Player 1 move:


## Summary

In this lecture we connected **software organization** (modules) with **recursion** and used both ideas to sketch a simple checkers AI.

Key takeaways:

- **Why Python:** interpreted and interactive, which suits exploratory work; and a "glue" language over fast compiled libraries — you steer the computation in Python and let packages like numpy do the heavy lifting in compiled code.

- **Modules and imports**
  - `import module` loads code from a file (like `checkers.py`) and gives you access via `module.name`.
  - `from module import name` imports specific names into your current namespace.
  - `importlib.reload(module)` is useful during development when you edit a module and want to re-run it without restarting the kernel.

- **Recursion**
  - A recursive function needs a **base case** (when to stop) and a **recursive step** (a smaller version of the same problem).
  - Recursion is a natural fit for **tree-shaped data** (nested lists, game trees).
  - Python has a recursion depth limit; deep recursion will eventually raise an error.

- **A simple game-tree search**
  - Generate possible moves, then possible responses, building a **tree of possibilities**.
  - Use a **scoring function** (here: piece count difference) to evaluate positions.
  - Use `copy.deepcopy` when you need an independent copy of a nested structure (like a board).

Limitations / next steps:

- A real game-playing agent usually uses **minimax** (with alpha–beta pruning) and a better evaluation function.
- Checkers also has extra rules (forced captures, king pieces, multi-jumps) that a full implementation must handle.
